In [26]:
# 生成 adam standard 版本

import os
import re
import shutil
import xml.etree.ElementTree as ET


DEL_JOINTS_NAME = ["wristRoll_Left", "wristRoll_Right"]


sp_tree = ET.parse("../adam_sp/adam_sp.urdf")
sp_root = sp_tree.getroot()

def remove_joint_and_children(root, target_joint_name):
    joints = {j.attrib['name']: j for j in root.findall('joint')}
    links = {l.attrib['name']: l for l in root.findall('link')}
    joints_to_remove = set()
    links_to_remove = set()

    def find_children(joint_name):
        if joint_name in joints:
            joints_to_remove.add(joint_name)
            child_link = joints[joint_name].find('child').attrib['link']
            links_to_remove.add(child_link)
            for j in joints.values():
                if j.find('parent') is not None and j.find('parent').attrib['link'] == child_link:
                    find_children(j.attrib['name'])

    find_children(target_joint_name)
    for joint in root.findall('joint'):
        if joint.attrib['name'] in joints_to_remove:
            root.remove(joint)
    for link in root.findall('link'):
        if link.attrib['name'] in links_to_remove:
            root.remove(link)

for joint_name in DEL_JOINTS_NAME:
    remove_joint_and_children(sp_root, joint_name)

new_folder = "../adam_standard/"
if os.path.exists(new_folder):
    shutil.rmtree(new_folder)
os.makedirs(new_folder, exist_ok=True)
sp_root.attrib["name"] = "adam_standard"
ET.indent(sp_tree, space="  ")
sp_tree.write("../adam_standard/adam_standard.urdf", encoding="utf-8", xml_declaration=True)


cur_dir = os.listdir(".")
pattern = re.compile(r'^meshes.*')
mesh_dirs = [d for d in cur_dir if pattern.match(d) and os.path.isdir(d)]
[shutil.copytree(d, f"../adam_standard/{d}") for d in mesh_dirs]


# open adam_sp.xml
sp_xml_path = "../adam_sp/adam_sp.xml"
DEL_MESHES_NAME = ["wristRollLeft", "wristRollRight"]
sp_xml_tree = ET.parse(sp_xml_path)
sp_xml_root = sp_xml_tree.getroot()
# modify the name of the robot
sp_xml_root.attrib["model"] = "adam_standard"
asset_tag = sp_xml_root.find("asset")
if asset_tag is not None:
    for tag in asset_tag.findall("mesh"):
        if tag.attrib["name"].startswith("L_") or tag.attrib["name"].startswith("R_") or tag.attrib["name"].find("wristRoll") != -1:
            asset_tag.remove(tag)

for body_name in DEL_MESHES_NAME:
    body = sp_xml_root.find(f".//body[@name='{body_name}']")
    if body is not None:
        parent = None
        for elem in sp_xml_root.iter():
            if body in list(elem):
                parent = elem
                break
        parent.remove(body)

actuator = sp_xml_root.find("actuator")
for motor in actuator.findall("motor"):
    if motor.attrib.get("joint").startswith("L_") or motor.attrib.get("joint").startswith("R_") or motor.attrib.get("joint").find("wristRoll") != -1:
        actuator.remove(motor)

sensor = sp_xml_root.find("sensor")
for s in list(sensor):
    if s.attrib.get("name").find("L_") != -1 or s.attrib.get("name").find("R_") != -1 or s.attrib.get("name").find("wristRoll") != -1:
        sensor.remove(s)
    
sp_xml_tree.write("../adam_standard/adam_standard.xml", encoding="utf-8", xml_declaration=True)

sp_scene_xml_path = "../adam_sp/scene_adam_sp.xml"
sp_scene_xml_tree = ET.parse(sp_scene_xml_path)
sp_scene_xml_root = sp_scene_xml_tree.getroot()
sp_scene_xml_root.attrib["model"] = "adam_standard scene"
include_tag = sp_scene_xml_root.find("include")
include_tag.attrib["file"] = "adam_standard.xml"
sp_scene_xml_tree.write("../adam_standard/scene_adam_standard.xml", encoding="utf-8", xml_declaration=True)